
# 02b — Fireline Version Resolution

This notebook resolves the remaining Fireline archive-history problem before NIROPS temporal analysis.

It tests whether `CreateDate` can identify a logical fireline lineage, compares multiple historical-version resolution rules, preserves historical incident coverage, and writes a new version-resolved GeoParquet dataset.

**Do not use the previous `GDB_TO_DATE is null` canonical file for downstream analysis.**


In [ ]:
import os  # Configurable project root for the public release.

from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

FIRELINE_GDB = (
    PROJECT_ROOT / "data" / "interim" / "joinability_audit"
    / "fireline_extracted" / "Data" / "FirelineEngagement.gdb"
)
FIRELINE_LAYER = "Firelines_Engagement_17_24"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "fireline_version_resolution"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

assert FIRELINE_GDB.exists(), FIRELINE_GDB

print("GDB:", FIRELINE_GDB)
print("Output:", OUTPUT_ROOT)


In [ ]:

import sys, subprocess, importlib.util

pkgs = {
    "pandas": "pandas",
    "geopandas": "geopandas",
    "pyogrio": "pyogrio",
    "shapely": "shapely",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
}

for module, package in pkgs.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import re, hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import normalize as shapely_normalize
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print("Python:", sys.executable)


## 1. Load and normalize the Fireline layer

In [ ]:

fireline = gpd.read_file(
    FIRELINE_GDB,
    layer=FIRELINE_LAYER,
    engine="pyogrio"
)

def normalize_irwin(v):
    if pd.isna(v):
        return None
    s = re.sub(r"[^A-Z0-9]", "", str(v).strip().upper())
    return s or None

def normalize_engagement(v):
    if pd.isna(v):
        return "Other/Unknown"
    s = re.sub(r"[^a-z]+", "", str(v).lower())
    if "notengag" in s:
        return "Not Engaged"
    if "burn" in s and "over" in s:
        return "Burned Over"
    if "held" in s or "hold" in s:
        return "Held"
    return "Other/Unknown"

fireline["_irwin_norm"] = fireline["IRWINID"].map(normalize_irwin)
fireline["_engagement_norm"] = fireline["FirelineEngagement"].map(normalize_engagement)

for c in ["CreateDate", "DateCurrent", "GDB_FROM_DATE", "GDB_TO_DATE", "LineDateTime"]:
    if c in fireline.columns:
        fireline[c] = pd.to_datetime(fireline[c], errors="coerce", utc=True)

print("Rows:", f"{len(fireline):,}")
print("Unique IRWIN:", f"{fireline['_irwin_norm'].nunique(dropna=True):,}")
print("CRS:", fireline.crs)


## 2. Create exact geometry fingerprints and physical line lengths

In [ ]:

def geom_hash(geom):
    if geom is None or geom.is_empty:
        return None
    try:
        geom = shapely_normalize(geom)
    except Exception:
        pass
    return hashlib.sha1(geom.wkb).hexdigest()

fireline["_geom_hash"] = [
    geom_hash(g)
    for g in tqdm(fireline.geometry, total=len(fireline), desc="Geometry hashes")
]

projected = fireline.to_crs(5070)
fireline["_length_km_geom"] = projected.geometry.length / 1000.0
fireline["_length_km_source"] = pd.to_numeric(
    fireline.get("LineLengthGeodesicKM"),
    errors="coerce"
)

print("Unique exact geometries:", f"{fireline['_geom_hash'].nunique(dropna=True):,}")
print("Geometry-derived total:", f"{fireline['_length_km_geom'].sum():,.2f} km")
print("Source-field total:", f"{fireline['_length_km_source'].sum():,.2f} km")



## 3. Test whether `IRWINID + CreateDate` behaves like a logical fireline ID

If one `IRWIN + CreateDate` group frequently contains many different geometries, `CreateDate` cannot safely be used by itself to collapse edited versions.


In [ ]:

create_reuse = (
    fireline
    .groupby(["_irwin_norm", "CreateDate"], dropna=False)
    .agg(
        rows=("_geom_hash", "size"),
        geometries=("_geom_hash", "nunique"),
        categories=("FeatureCategory", "nunique"),
        engagements=("_engagement_norm", "nunique"),
        first_gdb_from=("GDB_FROM_DATE", "min"),
        last_gdb_from=("GDB_FROM_DATE", "max"),
    )
    .reset_index()
)

pct_single_geometry = (create_reuse["geometries"] == 1).mean() * 100

reuse_summary = pd.DataFrame([
    ["IRWIN+CreateDate groups", len(create_reuse)],
    ["groups with 1 geometry", int((create_reuse["geometries"] == 1).sum())],
    ["groups with 2 geometries", int((create_reuse["geometries"] == 2).sum())],
    ["groups with >=3 geometries", int((create_reuse["geometries"] >= 3).sum())],
    ["maximum geometries in one group", int(create_reuse["geometries"].max())],
    ["percent with exactly 1 geometry", pct_single_geometry],
], columns=["metric", "value"])

display(reuse_summary)

create_reuse.sort_values(
    ["geometries", "rows"], ascending=False
).head(100).to_csv(
    OUTPUT_ROOT / "createdate_collision_candidates.csv",
    index=False
)

reuse_summary.to_csv(OUTPUT_ROOT / "createdate_reuse_summary.csv", index=False)


## 4. Compare historical-version resolution strategies

In [ ]:

def latest_per_group(df, keys, time_col):
    return (
        df.sort_values(keys + [time_col], na_position="first")
        .drop_duplicates(subset=keys, keep="last")
        .copy()
    )

strategies = {
    "S1_latest_gdbfrom_irwin_geom": latest_per_group(
        fireline, ["_irwin_norm", "_geom_hash"], "GDB_FROM_DATE"
    ),
    "S2_latest_datecurrent_irwin_geom": latest_per_group(
        fireline, ["_irwin_norm", "_geom_hash"], "DateCurrent"
    ),
    "S3_latest_gdbfrom_irwin_createdate": latest_per_group(
        fireline, ["_irwin_norm", "CreateDate"], "GDB_FROM_DATE"
    ),
    "S4_latest_datecurrent_irwin_createdate": latest_per_group(
        fireline, ["_irwin_norm", "CreateDate"], "DateCurrent"
    ),
    "S5_latest_gdbfrom_irwin_createdate_category": latest_per_group(
        fireline, ["_irwin_norm", "CreateDate", "FeatureCategory"], "GDB_FROM_DATE"
    ),
}

strategies["S6_exact_identity_dedup"] = (
    fireline.sort_values(["GDB_FROM_DATE", "DateCurrent"], na_position="first")
    .drop_duplicates(
        subset=["_irwin_norm", "_engagement_norm", "FeatureCategory", "_geom_hash"],
        keep="last"
    )
    .copy()
)

rows = []
raw_incidents = fireline["_irwin_norm"].nunique(dropna=True)

for name, df in strategies.items():
    cls = df["_engagement_norm"].value_counts()
    rows.append({
        "strategy": name,
        "rows": len(df),
        "unique_irwin": df["_irwin_norm"].nunique(dropna=True),
        "unique_geometry": df["_geom_hash"].nunique(dropna=True),
        "incident_retention_pct": df["_irwin_norm"].nunique(dropna=True) / raw_incidents * 100,
        "held": int(cls.get("Held", 0)),
        "burned_over": int(cls.get("Burned Over", 0)),
        "not_engaged": int(cls.get("Not Engaged", 0)),
        "geometry_length_km": df["_length_km_geom"].sum(),
        "source_length_km": df["_length_km_source"].sum(),
        "rows_per_geometry": len(df) / max(df["_geom_hash"].nunique(dropna=True), 1),
    })

strategy_comparison = pd.DataFrame(rows)
display(strategy_comparison)
strategy_comparison.to_csv(
    OUTPUT_ROOT / "version_resolution_strategy_comparison.csv",
    index=False
)



## 5. Conservative automatic recommendation

- If `IRWIN + CreateDate` is almost always one geometry, use the lineage-based strategy.
- Otherwise, use latest `GDB_FROM_DATE` per `IRWIN + exact geometry`.

The latter is intentionally conservative because it resolves archive copies without merging spatially distinct lines.


In [ ]:

if pct_single_geometry >= 99.5:
    RECOMMENDED_STRATEGY = "S3_latest_gdbfrom_irwin_createdate"
    recommendation_reason = (
        "IRWIN + CreateDate behaves almost uniquely as a logical feature lineage key."
    )
else:
    RECOMMENDED_STRATEGY = "S1_latest_gdbfrom_irwin_geom"
    recommendation_reason = (
        "CreateDate is reused across multiple geometries often enough that collapsing "
        "solely by IRWIN + CreateDate could merge distinct firelines. Latest "
        "GDB_FROM_DATE per IRWIN + exact geometry is safer."
    )

print("Recommended:", RECOMMENDED_STRATEGY)
print("Reason:", recommendation_reason)

# Optional manual override after reviewing the tables:
# RECOMMENDED_STRATEGY = "S1_latest_gdbfrom_irwin_geom"

canonical = strategies[RECOMMENDED_STRATEGY].copy()


## 6. Canonical outcome and quality summaries

In [ ]:

canonical_outcomes = (
    canonical.groupby("_engagement_norm", dropna=False)
    .agg(
        rows=("_engagement_norm", "size"),
        unique_geometries=("_geom_hash", "nunique"),
        unique_incidents=("_irwin_norm", "nunique"),
        geometry_length_km=("_length_km_geom", "sum"),
        source_length_km=("_length_km_source", "sum"),
    )
    .reset_index()
    .rename(columns={"_engagement_norm": "engagement"})
)

quality = pd.DataFrame([
    ["canonical rows", len(canonical)],
    ["unique IRWIN", canonical["_irwin_norm"].nunique(dropna=True)],
    ["unique geometry", canonical["_geom_hash"].nunique(dropna=True)],
    ["incident retention %", canonical["_irwin_norm"].nunique(dropna=True) / raw_incidents * 100],
    ["invalid geometry rows", int((~canonical.geometry.is_valid & canonical.geometry.notna()).sum())],
    ["missing normalized IRWIN rows", int(canonical["_irwin_norm"].isna().sum())],
])

display(canonical_outcomes)
display(quality)

canonical_outcomes.to_csv(OUTPUT_ROOT / "canonical_outcome_summary.csv", index=False)
quality.to_csv(OUTPUT_ROOT / "canonical_quality_summary.csv", index=False)


## 7. Save the new version-resolved dataset

In [ ]:

canonical_path = PROCESSED_ROOT / "fireline_qaqc_version_resolved.parquet"

canonical.to_parquet(canonical_path, index=False)

print("Saved:", canonical_path)
print("Size:", f"{canonical_path.stat().st_size / (1024**2):,.2f} MB")

incident_inventory = (
    canonical.groupby("_irwin_norm", dropna=False)
    .agg(
        incident_name=("IncidentName", "first"),
        year=("Year", "first"),
        rows=("_geom_hash", "size"),
        unique_geometries=("_geom_hash", "nunique"),
        geometry_length_km=("_length_km_geom", "sum"),
    )
    .reset_index()
)

incident_inventory.to_csv(
    OUTPUT_ROOT / "canonical_incident_inventory.csv",
    index=False
)


## 8. Write the version-resolution report

In [ ]:

def to_md(df, n=100):
    if df is None or len(df) == 0:
        return "_No rows._"
    try:
        return df.head(n).to_markdown(index=False)
    except Exception:
        return df.head(n).to_string(index=False)

report = f"""# Fireline Version Resolution Report

Generated: {datetime.now(timezone.utc).isoformat()}

## Source

- Raw rows: **{len(fireline):,}**
- Raw unique IRWIN IDs: **{raw_incidents:,}**
- Raw unique exact geometries: **{fireline['_geom_hash'].nunique(dropna=True):,}**

## CreateDate lineage diagnostic

{to_md(reuse_summary)}

`IRWIN + CreateDate` groups with exactly one geometry: **{pct_single_geometry:.2f}%**

## Strategy comparison

{to_md(strategy_comparison)}

## Recommended strategy

**{RECOMMENDED_STRATEGY}**

{recommendation_reason}

## Canonical outcomes

{to_md(canonical_outcomes)}

## Canonical quality summary

{to_md(quality)}

## Canonical file

`{canonical_path}`

## Important interpretation

The prior `GDB_TO_DATE is null` provisional canonical dataset should not be used downstream because it removed a large share of historical incidents.

This version-resolved output prioritizes preservation of historical incident coverage.

For spatial analyses, use geometry-derived lengths until the discrepancy with `LineLengthGeodesicKM` is fully explained.

## Next step

Re-run the Fireline × NIROPS exact incident join using this version-resolved dataset, then begin spatial-temporal corroboration of Held and Burned Over lines using NIROPS progression observations.
"""

report_path = OUTPUT_ROOT / "FIRELINE_VERSION_RESOLUTION_REPORT.md"
report_path.write_text(report, encoding="utf-8")

print(report)
print("\nSaved:", report_path)


## 9. Final output inventory

In [ ]:

print("OUTPUTS")
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p.name:<50} {p.stat().st_size / 1024:>10,.2f} KB")

print("\nCANONICAL")
print(canonical_path)
print("\nDone.")
